# 学术文献真实性验证工具 — 演示## AI推荐的文献是真是假？

## 1. 导入验证引擎

In [ ]:
import sys, jsonsys.path.insert(0, ".")from verify_engine import ReferenceRecord, verify_record, verify_batchimport pandas as pd

## 2. 单篇验证：真实文献 (Nature 2013)

In [ ]:
ref = ReferenceRecord(    title="Nanometre-scale thermometry in a living cell",    authors="G. Kucsko, P. C. Maurer",    journal="Nature",    doi="10.1038/nature12373",    year=2013)result = verify_record(ref)print(f"状态: {result.status}, 可信度: {result.score}/100")for d in result.details:    print(f"  {d}")

**预期结果**: 状态=可靠, 得分≥90。DOI格式正确→+10, 年份合理→+10, CrossRef DOI匹配→+40, OpenAlex确认→+30, 元数据一致→+10。

## 3. 单篇验证：虚假文献 (伪造DOI)

In [ ]:
ref2 = ReferenceRecord(    title="A Novel Quantum Computing Approach for Solving NP-Complete Problems in Linear Time",    authors="Fictitious Author",    doi="10.9999/fake.paper.2024",    year=2024)result2 = verify_record(ref2)print(f"状态: {result2.status}, 可信度: {result2.score}/100")for d in result2.details:    print(f"  {d}")

**预期结果**: 状态=虚假, 得分≤35。DOI格式有效但数据库无记录→+10, 年份合理→+10, 搜索不匹配→+0。

## 4. 无DOI的经典论文验证 (arXiv论文)

In [ ]:
ref3 = ReferenceRecord(    title="Attention Is All You Need",    authors="Vaswani et al.",    journal="NIPS",    year=2017)result3 = verify_record(ref3)print(f"状态: {result3.status}, 可信度: {result3.score}/100")for d in result3.details:    print(f"  {d}")

**预期结果**: 状态=可靠 或 可疑。无DOI需要标题搜索，可能匹配到arxiv版本。

## 5. 批量验证演示数据

In [ ]:
df = pd.read_csv("data/demo_data.csv")print(f"加载 {len(df)} 条记录")df.head()

## 6. 批量验证结果统计

In [ ]:
records = []for _, row in df.iterrows():    records.append(ReferenceRecord(        title=str(row["title"]),        authors=str(row.get("authors", "")),        journal=str(row.get("journal", "")) if pd.notna(row.get("journal")) else None,        doi=str(row.get("doi", "")) if pd.notna(row.get("doi")) else None,        year=int(row["year"]) if pd.notna(row.get("year")) else None,    ))results = []for i, ref in enumerate(records):    print(f"验证 {i+1}/{len(records)}: {ref.title[:50]}...")    r = verify_record(ref)    results.append({        "标题": ref.title,        "DOI": ref.doi or "",        "状态": r.status,        "可信度": r.score,        "详情": " | ".join(r.details),    })result_df = pd.DataFrame(results)result_df[['标题', 'DOI', '状态', '可信度']]

## 7. 结果汇总

In [ ]:
status_counts = result_df["状态"].value_counts()print("=== 验证结果汇总 ===")print(f"可靠: {status_counts.get("可靠", 0)}")print(f"可疑: {status_counts.get("可疑", 0)}")print(f"虚假: {status_counts.get("虚假", 0)}")# 用简单条形图展示result_df["状态"].value_counts().plot.bar(color=["green", "orange", "red"], title="验证结果分布")

## 8. 解读：AI文献虚假特征

通过本工具可以发现 AI 生成的虚假文献常见特征：1. **DOI格式正确但数据库无记录** — DOI后缀是乱编的2. **标题搜索返回不相关结果** — 标题中的关键词组合现实中不存在3. **作者与期刊领域不匹配** — 捏造的作者名在真实期刊中无记录4. **过于夸张的标题** — "100% accuracy"、"linear time NP-complete"等违反学术常识5. **期刊名称拼写错误或不存在** — 虚构的期刊名本工具的评分维度能有效捕捉这些特征。